In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
sandbox = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/sandbox_solution.csv"
)

print(sandbox.shape)
sandbox.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.143129
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.102447
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.102447
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.147267
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.147267


In [3]:
submission = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/outputs/submissions/submission_best_linear_interpolation.csv"
)

print(submission.shape)
submission.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.11373
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.10150
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.16344
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.09681
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.10730


In [4]:
print(
    (sandbox["id"] == submission["id"]).all()
)

False


In [5]:
print("Sandbox:")
print(sandbox.head())

print("\nSubmission:")
print(submission.head())

Sandbox:
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.147267
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.147267

Submission:
                                      id    value
0  07-01-2026 09:15||NIFTY27JAN2625500CE  0.11373
1  07-01-2026 09:15||NIFTY27JAN2625800CE  0.10150
2  07-01-2026 09:15||NIFTY27JAN2624100PE  0.16344
3  07-01-2026 09:20||NIFTY27JAN2625300CE  0.09681
4  07-01-2026 09:20||NIFTY27JAN2625400CE  0.10730


In [6]:
print("Sandbox columns:")
print(sandbox.columns)

print("\nSubmission columns:")
print(submission.columns)

Sandbox columns:
Index(['id', 'value'], dtype='object')

Submission columns:
Index(['id', 'value'], dtype='object')


In [7]:
print("Sandbox rows:", len(sandbox))
print("Submission rows:", len(submission))

Sandbox rows: 5460
Submission rows: 5460


In [8]:
sandbox_ids = set(sandbox["id"])
submission_ids = set(submission["id"])

print("IDs in both:", len(sandbox_ids & submission_ids))
print("Sandbox only:", len(sandbox_ids - submission_ids))
print("Submission only:", len(submission_ids - sandbox_ids))

IDs in both: 5460
Sandbox only: 0
Submission only: 0


In [9]:
merged = sandbox.merge(
    submission,
    on="id",
    suffixes=("_true", "_pred")
)

print(merged.shape)
merged.head()

(5460, 3)


,id,value_true,value_pred
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.143129,0.163440
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.102447,0.113730
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.102447,0.101500
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.147267,0.170055
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.147267,0.159770


In [10]:
import numpy as np

mse = np.mean(
    (merged["value_true"] - merged["value_pred"]) ** 2
)

print("Real MSE =", mse)

Real MSE = 0.01979902765469819


In [11]:
merged["sq_error"] = (
    merged["value_true"] - merged["value_pred"]
) ** 2

worst = merged.sort_values(
    "sq_error",
    ascending=False
)

worst.head(20)

,id,value_true,value_pred,sq_error
5459,27-01-2026 15:25||NIFTY27JAN2626500CE,1.743962,4.815900,9.436803
5455,27-01-2026 15:25||NIFTY27JAN2625100PE,3.403524,0.687310,7.377817
5458,27-01-2026 15:25||NIFTY27JAN2626400CE,1.743962,4.458130,7.366708
5453,27-01-2026 15:25||NIFTY27JAN2623800PE,3.403524,5.794990,5.719111
5457,27-01-2026 15:25||NIFTY27JAN2626300CE,1.743962,4.100360,5.552612
5446,27-01-2026 15:20||NIFTY27JAN2623800PE,1.988463,3.944630,3.826590
5454,27-01-2026 15:25||NIFTY27JAN2624900PE,3.403524,1.490110,3.661152
5452,27-01-2026 15:20||NIFTY27JAN2626400CE,1.674703,3.381057,2.911643
5447,27-01-2026 15:20||NIFTY27JAN2623900PE,1.988463,3.674480,2.842654
5456,27-01-2026 15:25||NIFTY27JAN2626100CE,1.743962,3.384820,2.692415


In [12]:
worst["datetime"] = worst["id"].str.split("||").str[0]

worst["strike"] = worst["id"].str.split("||").str[1]

print(
    worst["datetime"]
    .value_counts()
    .head(20)
)

datetime
    5460
Name: count, dtype: int64


In [13]:
merged["datetime"] = merged["id"].str.split("||").str[0]

time_error = (
    merged.groupby("datetime")["sq_error"]
    .mean()
    .sort_values(ascending=False)
)

print(time_error.head(30))

datetime
    0.019799
Name: sq_error, dtype: float64


In [14]:
print(merged["id"].head())

0    07-01-2026 09:15||NIFTY27JAN2624100PE
1    07-01-2026 09:15||NIFTY27JAN2625500CE
2    07-01-2026 09:15||NIFTY27JAN2625800CE
3    07-01-2026 09:20||NIFTY27JAN2624000PE
4    07-01-2026 09:20||NIFTY27JAN2624200PE
Name: id, dtype: object


In [15]:
test = merged["id"].str.split("||", expand=True)

print(test.head())

  0  1  2  3  4  5  6  7  8  9   ... 29 30 31 32 33 34 35 36 37 38
0     0  7  -  0  1  -  2  0  2  ...  2  6  2  4  1  0  0  P  E   
1     0  7  -  0  1  -  2  0  2  ...  2  6  2  5  5  0  0  C  E   
2     0  7  -  0  1  -  2  0  2  ...  2  6  2  5  8  0  0  C  E   
3     0  7  -  0  1  -  2  0  2  ...  2  6  2  4  0  0  0  P  E   
4     0  7  -  0  1  -  2  0  2  ...  2  6  2  4  2  0  0  P  E   

[5 rows x 39 columns]


In [16]:
merged["datetime"] = merged["id"].str.split(r"\|\|").str[0]
merged["strike"] = merged["id"].str.split(r"\|\|").str[1]

print(merged["datetime"].head())
print(merged["strike"].head())

0    07-01-2026 09:15
1    07-01-2026 09:15
2    07-01-2026 09:15
3    07-01-2026 09:20
4    07-01-2026 09:20
Name: datetime, dtype: object
0    NIFTY27JAN2624100PE
1    NIFTY27JAN2625500CE
2    NIFTY27JAN2625800CE
3    NIFTY27JAN2624000PE
4    NIFTY27JAN2624200PE
Name: strike, dtype: object


In [17]:
print(merged["id"].head())

0    07-01-2026 09:15||NIFTY27JAN2624100PE
1    07-01-2026 09:15||NIFTY27JAN2625500CE
2    07-01-2026 09:15||NIFTY27JAN2625800CE
3    07-01-2026 09:20||NIFTY27JAN2624000PE
4    07-01-2026 09:20||NIFTY27JAN2624200PE
Name: id, dtype: object


In [18]:
test = merged["id"].str.split(r"\|\|", expand=True)
print(test.head())

                  0                    1
0  07-01-2026 09:15  NIFTY27JAN2624100PE
1  07-01-2026 09:15  NIFTY27JAN2625500CE
2  07-01-2026 09:15  NIFTY27JAN2625800CE
3  07-01-2026 09:20  NIFTY27JAN2624000PE
4  07-01-2026 09:20  NIFTY27JAN2624200PE


In [19]:
print(test[0].nunique())
print(test[1].nunique())

972
28


In [20]:
merged["datetime"] = merged["id"].str.split(r"\|\|").str[0]
merged["strike"] = merged["id"].str.split(r"\|\|").str[1]

In [21]:
time_error = (
    merged.groupby("datetime")["sq_error"]
    .mean()
    .sort_values(ascending=False)
)

print(time_error.head(20))

datetime
27-01-2026 15:25    5.972374
27-01-2026 15:20    1.936847
27-01-2026 15:15    0.948049
27-01-2026 15:00    0.621461
27-01-2026 15:10    0.562911
27-01-2026 15:05    0.526112
27-01-2026 14:50    0.485961
27-01-2026 14:55    0.412374
27-01-2026 14:45    0.381459
27-01-2026 14:40    0.335033
27-01-2026 14:35    0.286837
27-01-2026 14:25    0.277321
27-01-2026 14:30    0.249979
27-01-2026 14:20    0.184729
27-01-2026 13:55    0.153424
27-01-2026 14:00    0.149829
27-01-2026 14:10    0.146572
27-01-2026 13:50    0.145394
27-01-2026 13:25    0.143399
27-01-2026 14:05    0.124855
Name: sq_error, dtype: float64


In [22]:
strike_error = (
    merged.groupby("strike")["sq_error"]
    .mean()
    .sort_values(ascending=False)
)

print(strike_error.head(20))

strike
NIFTY27JAN2623800PE    0.075953
NIFTY27JAN2626400CE    0.073867
NIFTY27JAN2626500CE    0.066174
NIFTY27JAN2625100PE    0.054063
NIFTY27JAN2626300CE    0.048542
NIFTY27JAN2624900PE    0.036090
NIFTY27JAN2623900PE    0.034699
NIFTY27JAN2624000PE    0.032248
NIFTY27JAN2625000PE    0.019787
NIFTY27JAN2626100CE    0.016549
NIFTY27JAN2625400CE    0.014105
NIFTY27JAN2624100PE    0.010742
NIFTY27JAN2625200CE    0.010437
NIFTY27JAN2625500CE    0.006999
NIFTY27JAN2626200CE    0.006939
NIFTY27JAN2624200PE    0.006908
NIFTY27JAN2624800PE    0.006775
NIFTY27JAN2625300CE    0.006241
NIFTY27JAN2626000CE    0.005267
NIFTY27JAN2625600CE    0.004149
Name: sq_error, dtype: float64


In [23]:
late_expiry = merged[
    merged["datetime"].str.startswith("27-01-2026")
]

print("Rows:", len(late_expiry))

print(
    "Error share:",
    late_expiry["sq_error"].sum() /
    merged["sq_error"].sum()
)

Rows: 408
Error share: 0.9693233883688132


In [26]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

print(train.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(975, 30)


In [27]:
train["datetime"] = pd.to_datetime(
    train["datetime"],
    format="%d-%m-%Y %H:%M"
)

option_cols = train.columns[2:]

train["row_mean_iv"] = train[option_cols].mean(axis=1)

late = train[
    train["datetime"].dt.strftime("%d-%m-%Y")
    == "27-01-2026"
]

late[[
    "datetime",
    "row_mean_iv"
]].tail(20)

,datetime,row_mean_iv
955,2026-01-27 13:50:00,0.827916
956,2026-01-27 13:55:00,0.834246
957,2026-01-27 14:00:00,0.883073
958,2026-01-27 14:05:00,0.820691
959,2026-01-27 14:10:00,0.847961
960,2026-01-27 14:15:00,0.934051
961,2026-01-27 14:20:00,0.879810
962,2026-01-27 14:25:00,0.979321
963,2026-01-27 14:30:00,1.015733
964,2026-01-27 14:35:00,1.061915
